# Llama 3.2 QLoRA — hyperparameter search and full Kaggle training (v2: retrieved-context training)

**What v2 changes (everything else is identical to `train_llama_qlora_kaggle.ipynb`):** the training CSV may carry a `context_json` column. It holds the passages the model is *shown* (realistic retrieval, including near-miss passages), while `evidence_json` stays the passages the answer *cites*. Escalation tickets keep empty evidence and are flagged unanswerable again. This teaches the model that related-looking context does not by itself mean there is an answer. A CSV without `context_json` behaves exactly as before. The defaults are a full run with the sprint-3 screened settings (learning rate 2e-4, `adamw_8bit`, LoRA rank 16), so the only experimental change versus the first full run is the data.

This is the Kaggle-native companion to `train_llama_qlora.ipynb`. It keeps the same ticket validation, batch-based splits, completion masking, training, plots, and adapter/checkpoint audits. There are **no Colab imports, upload widgets, Drive mounts, or hard-coded absolute paths**; see the steps below.

1. Import this notebook into Kaggle. Keep the notebook and any attached datasets private.
2. Upload `train_tickets.csv` as a private Kaggle Dataset and attach it with **Add Input**. Only use the training CSV that excludes the held-out tenant.
3. In notebook settings, select an available NVIDIA GPU and enable Internet for package/model downloads. Check your account's remaining GPU quota and session limits.
4. Choose **Save Version → Save & Run All** to launch a fresh background execution, rather than relying on an interactive editor session. Confirm the background job starts and passes setup before leaving. It runs on Kaggle's servers; your laptop can be closed.
5. After the version completes, download the output folder from the version's **Output** section. Outputs are under `/kaggle/working/Capstone_Models/<RUN_NAME>/`.


## 1. Install in the Kaggle worker

Run in a fresh Kaggle GPU session. Internet must be enabled for installation and downloading the Hugging Face model. The resolved package versions are saved with every run. This installation has not been executed on a Kaggle GPU by the notebook author; use a smoke run to verify the worker environment before a long job.

Do not add notebook widgets, manual `input()` calls, `files.upload()`, Drive mounting, or interactive authentication to a background execution. Use attached datasets and optional Kaggle Secrets instead. If installation requires a restart, resolve it in an interactive test first; do not leave a background job waiting for a restart.

The install deliberately omits blanket `--upgrade`: retain installed packages when they meet requirements. Pandas 2.2.3 matches the Colab requirement reported in the tested installation log; Matplotlib 3.10.0 avoids an unnecessary 3.11 upgrade, and Bottleneck >=1.5.0 prevents the reported optional pandas accelerator warning. These pins do not resolve every unrelated package conflict in the hosted image (for example Gradio, cloud filesystem clients, or profiling tools). This notebook uses local CSV/filesystem paths, not those integrations.

Unsloth patching banners and the Python/PyTorch/GPU inventory are informational. A Bottleneck `UserWarning` is not a training traceback. An installation log ending with `Successfully installed` can still report incompatible preinstalled packages: inspect those reports rather than treating them as proof of either training success or failure. Verify model loading and a smoke trial in the actual worker. Do not install or upgrade packages during training, or change a checkpoint's environment just to silence warnings.


In [ ]:
%pip install unsloth "transformers==4.57.6" "trl==0.22.2" "datasets==4.3.0" "pandas==2.2.3" "matplotlib==3.10.0" "bottleneck>=1.5.0" wandb


## 2. Choose search, full training, or smoke testing

**Hyperparameter search is training**, not inference tuning. An optimizer (such as AdamW) is one of the training settings you can compare; grid/random search is the process for comparing combinations.

The default `RUN_MODE="search"` runs four trials: two learning rates × two optimizers. Every trial starts from fresh base weights with the same seed, the same 256 representative training tickets, the same 64 validation tickets, the same task mix, and 60 optimizer updates. Evaluation and saving happen every 30 steps. There is no automatic full run afterward.

Edit the lists in `SEARCH_SPACE` to compare learning rate, optimizer, LoRA rank/alpha, weight decay, or warmup. `grid` visits combinations in a deterministic order; `random` samples unique combinations reproducibly. `SEARCH_MAX_TRIALS` caps either mode. Batch size, accumulation, token length, seed, task mix, and update budget stay fixed so comparisons use the same examples and compute schedule. Expand rank and alpha deliberately: their ratio affects LoRA scaling.

The search minimizes **JSON-task validation completion loss** (`serving_label` when enabled, otherwise `label`), while training on the selected label/rationale mix. It is a screening proxy, not a guarantee of grounded responses, optimal full-run settings, or convergence. A small subset can rank settings differently from full data.

After search completes, attach its saved Kaggle output to a new notebook execution, set `RUN_MODE="full"`, `BEST_PARAMS_FILE="auto"` (or the exact attached JSON path), and a **new** `RUN_NAME`. Leave `RESUME=False`: this transfers settings, not the partially trained trial adapter. Full mode uses all training tickets, the complete validation split, and `NUM_TRAIN_EPOCHS`. Its validation curves use the full task mix; `validation_by_task.json` separates each task.

`RESUME=True` continues an interrupted run with its original name, data, and settings. In search mode, it skips completed trials and restarts failed/interrupted trials from fresh weights in a new attempt folder. Attach the entire prior run output and set `RESUME_SOURCE_DIR` for a fresh Kaggle worker. Do not change settings halfway through a resumed search. Existing Colab runs are unaffected.

Each experiment gets its own output directory. Keep the final held-out benchmark out of all hyperparameter selection.

In [ ]:
import os
from pathlib import Path

# This notebook intentionally uses one GPU, even on a T4 x2 allocation.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

RUN_NAME = "llama32_3b_hardneg_full_01"  # Use a new name for each independent experiment.
RUN_MODE = "full"  # "search", "full", or "smoke". v2 default: a full run with the screened sprint-3 settings.
RESUME = False
MODEL_NAME = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit"
HELD_OUT_TENANT = "harbor_view"
# Leave blank to discover a single attached train_tickets.csv automatically.
DATA_CSV = ""
WORK_ROOT = Path("/kaggle/working/Capstone_Models")
# Example: /kaggle/input/<attached-output>/Capstone_Models/<original-run-name>
RESUME_SOURCE_DIR = ""
ALLOW_ENVIRONMENT_CHANGE = False  # Only for intentional cross-environment migration.


SEED = 42
VALIDATION_FRACTION = 0.10
TASK_MODE = "label_and_rationale"
INCLUDE_SERVING_LABELS = True
MAX_SEQ_LENGTH = 2048
LORA_RANK = 16
LORA_ALPHA = 16
LEARNING_RATE = 2e-4  # Winner of the sprint-3 screening search (trial_003); no new search is needed.
OPTIMIZER = "adamw_8bit"
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.03
NUM_TRAIN_EPOCHS = 1
TRAIN_BATCH_SIZE = 1  # Conservative starting point for a T4.
GRADIENT_ACCUMULATION_STEPS = 8
SMOKE_MAX_TICKETS = 128
SMOKE_MAX_STEPS = 20
FULL_EVAL_SAVE_STEPS = 200
USE_WANDB = False  # Optional: enable only if you want remote experiment logging.
WANDB_PROJECT = "pro390-support-qlora"
EXPORT_GGUF = False  # Optional, after adapter/checkpoint verification.
GGUF_QUANTIZATIONS = ["q4_k_m"]  # For the full comparison: ["q4_k_m", "q5_k_m", "q8_0"].

# Full run: attach the search output and use "auto" or its exact JSON path.
BEST_PARAMS_FILE = ""
SEARCH_METHOD = "grid"  # "grid" or "random"
SEARCH_MAX_TRIALS = 4
SEARCH_TRAIN_TICKETS = 256
SEARCH_VALIDATION_TICKETS = 64
SEARCH_MAX_STEPS = 60
SEARCH_EVAL_STEPS = 30
SEARCH_SPACE = {
    "learning_rate": [1e-4, 2e-4],
    "optimizer": ["adamw_8bit", "adamw_torch"],
    "lora_rank": [16],
    "lora_alpha": [16],
    "weight_decay": [0.01],
    "warmup_ratio": [0.03],
}

assert RUN_MODE in {"search", "smoke", "full"}
assert SEARCH_METHOD in {"grid", "random"}
assert min(SEARCH_MAX_TRIALS, SEARCH_TRAIN_TICKETS, SEARCH_VALIDATION_TICKETS, SEARCH_MAX_STEPS, SEARCH_EVAL_STEPS) > 0
assert SEARCH_EVAL_STEPS <= SEARCH_MAX_STEPS
if RUN_MODE == "search" and USE_WANDB:
    raise ValueError("Search logs to local trial folders; set USE_WANDB=False.")
assert TASK_MODE in {"label_only", "label_and_rationale"}
assert RUN_NAME and Path(RUN_NAME).name == RUN_NAME and RUN_NAME not in {".", ".."}
assert 0 < VALIDATION_FRACTION < 1
assert min(MAX_SEQ_LENGTH, LORA_RANK, LORA_ALPHA, TRAIN_BATCH_SIZE,
           GRADIENT_ACCUMULATION_STEPS, SMOKE_MAX_TICKETS, SMOKE_MAX_STEPS,
           FULL_EVAL_SAVE_STEPS) > 0
assert LEARNING_RATE > 0 and NUM_TRAIN_EPOCHS > 0

In [ ]:
# Unsloth must be imported before Transformers/PEFT so its patches apply.
from unsloth import FastLanguageModel, is_bfloat16_supported
import torch
import os
import sys
import json
import math
import random
import hashlib
import time
import importlib.metadata
from datetime import datetime, timezone
from html import escape

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from datasets import Dataset
from huggingface_hub import login
from trl import SFTConfig, SFTTrainer
from transformers import TrainerCallback, set_seed
from transformers.trainer_utils import get_last_checkpoint
from unsloth.chat_templates import get_chat_template

if not torch.cuda.is_available():
    raise RuntimeError("Enable an NVIDIA GPU in Kaggle notebook settings, then start a fresh execution.")
set_seed(SEED)
print(f"Python: {sys.version.split()[0]} | PyTorch: {torch.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory / 2**30:.1f} GiB")
print("Precision:", "bfloat16" if is_bfloat16_supported() else "float16")


def optional_secret(name):
    value = os.environ.get(name)
    if value:
        return value
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        # Public model + disabled W&B require no credentials. Never log secrets.
        return None


hf_token = optional_secret("HF_TOKEN")
if hf_token:
    login(token=hf_token, add_to_git_credential=False)
else:
    print("HF_TOKEN is optional for this public Unsloth checkpoint. For gated Meta weights, grant model access and add an enabled Kaggle secret named HF_TOKEN.")

wandb_run = None
if USE_WANDB:
    import wandb
    wandb_key = optional_secret("WANDB_API_KEY")
    if not wandb_key:
        raise ValueError("Add WANDB_API_KEY to Kaggle Secrets, or set USE_WANDB=False.")
    wandb.login(key=wandb_key)

In [ ]:
def validate_hyperparameters(params):
    expected = {"learning_rate", "optimizer", "lora_rank", "lora_alpha", "weight_decay", "warmup_ratio"}
    if set(params) != expected:
        raise ValueError(f"Hyperparameter keys must be {sorted(expected)}.")
    if not isinstance(params["optimizer"], str) or params["optimizer"] not in {"adamw_8bit", "adamw_torch", "paged_adamw_8bit"}:
        raise ValueError("Supported optimizers: adamw_8bit, adamw_torch, paged_adamw_8bit.")
    for key in ("learning_rate", "weight_decay", "warmup_ratio"):
        if type(params[key]) not in (int, float) or not math.isfinite(params[key]):
            raise ValueError(f"{key} must be a finite number.")
    if params["learning_rate"] <= 0 or params["weight_decay"] < 0 or not 0 <= params["warmup_ratio"] < 1:
        raise ValueError("Invalid learning rate, weight decay, or warmup ratio.")
    for key in ("lora_rank", "lora_alpha"):
        if type(params[key]) is not int or params[key] <= 0:
            raise ValueError(f"{key} must be a positive integer.")
    return dict(params)


def build_search_plan(space, defaults, method, max_trials, seed):
    # Mixed-radix indexing avoids materializing a large Cartesian product.
    if method not in {"grid", "random"}:
        raise ValueError("SEARCH_METHOD must be 'grid' or 'random'.")
    if type(max_trials) is not int or max_trials <= 0:
        raise ValueError("SEARCH_MAX_TRIALS must be a positive integer.")
    if not space or set(space) - set(defaults):
        raise ValueError("SEARCH_SPACE contains an unknown hyperparameter or is empty.")
    keys = sorted(space)
    choices = []
    for key in keys:
        if not isinstance(space[key], list) or not space[key]:
            raise ValueError(f"SEARCH_SPACE[{key!r}] must be a nonempty list.")
        values = []
        for value in space[key]:
            validate_hyperparameters({**defaults, key: value})
            if value not in values:
                values.append(value)
        choices.append(values)
    total = math.prod(len(values) for values in choices)
    count = min(max_trials, total)
    indices = list(range(count)) if method == "grid" else random.Random(seed).sample(range(total), count)
    plan = []
    for trial_number, index in enumerate(indices, 1):
        remaining, selected = index, {}
        for key, values in reversed(list(zip(keys, choices))):
            remaining, position = divmod(remaining, len(values))
            selected[key] = values[position]
        plan.append({"trial_id": f"trial_{trial_number:03d}",
                     "hyperparameters": validate_hyperparameters({**defaults, **selected})})
    return plan, total


def stratified_ticket_subset(frame, max_tickets, seed):
    if type(max_tickets) is not int or max_tickets <= 0:
        raise ValueError("Subset size must be a positive integer.")
    groups = list(frame.groupby(["business", "action", "is_answerable"], sort=True))
    size = min(max_tickets, len(frame))
    if not groups or size < len(groups):
        raise ValueError(f"Subset needs at least {len(groups)} tickets to represent every business/action/answerability group.")
    ideal = [size * len(group) / len(frame) for _, group in groups]
    counts = [max(1, math.floor(value)) for value in ideal]
    while sum(counts) > size:
        index = max((i for i in range(len(groups)) if counts[i] > 1), key=lambda i: counts[i] - ideal[i])
        counts[index] -= 1
    while sum(counts) < size:
        index = max((i for i, (_, group) in enumerate(groups) if counts[i] < len(group)), key=lambda i: ideal[i] - counts[i])
        counts[index] += 1
    parts = [group.sample(n=count, random_state=seed + index)
             for index, ((_, group), count) in enumerate(zip(groups, counts))]
    return pd.concat(parts).sample(frac=1, random_state=seed).reset_index(drop=True)


def rank_search_results(results):
    eligible = [row for row in results if row.get("status") == "completed"
                and type(row.get("validation_loss")) in (int, float)
                and math.isfinite(row["validation_loss"]) and row["validation_loss"] >= 0]
    return sorted(eligible, key=lambda row: (row["validation_loss"], row["trial_id"]))


def load_promoted_parameters(path, defaults, expected_metadata):
    payload = json.loads(Path(path).read_text())
    if payload.get("search_complete") is not True:
        raise ValueError("The search did not finish. Review its trial results before promoting a configuration.")
    for key, expected in expected_metadata.items():
        if payload.get(key) != expected:
            raise ValueError(f"Search/full-run mismatch for {key}; use the same model, data split, and task format.")
    params = validate_hyperparameters(payload.get("hyperparameters", {}))
    return {**defaults, **params}, payload


BASE_HYPERPARAMETERS = validate_hyperparameters({
    "learning_rate": LEARNING_RATE, "optimizer": OPTIMIZER,
    "lora_rank": LORA_RANK, "lora_alpha": LORA_ALPHA,
    "weight_decay": WEIGHT_DECAY, "warmup_ratio": WARMUP_RATIO,
})
BEST_PARAMS_PAYLOAD = None
if BEST_PARAMS_FILE:
    if RUN_MODE != "full":
        raise ValueError("BEST_PARAMS_FILE is only for a full run; leave it blank for search/smoke.")
    if BEST_PARAMS_FILE == "auto":
        candidates = sorted(Path("/kaggle/input").rglob("best_hyperparameters.json"))
        if len(candidates) != 1:
            raise ValueError("Attach one search output or set BEST_PARAMS_FILE to an explicit file path.")
        BEST_PARAMS_FILE = str(candidates[0])
    BASE_HYPERPARAMETERS, BEST_PARAMS_PAYLOAD = load_promoted_parameters(
        BEST_PARAMS_FILE, BASE_HYPERPARAMETERS, {
            "model_name": MODEL_NAME, "held_out_tenant": HELD_OUT_TENANT,
            "seed": SEED, "validation_fraction": VALIDATION_FRACTION,
            "task_mode": TASK_MODE, "include_serving_labels": INCLUDE_SERVING_LABELS,
            "max_seq_length": MAX_SEQ_LENGTH,
        },
    )
    LEARNING_RATE = BASE_HYPERPARAMETERS["learning_rate"]
    OPTIMIZER = BASE_HYPERPARAMETERS["optimizer"]
    LORA_RANK = BASE_HYPERPARAMETERS["lora_rank"]
    LORA_ALPHA = BASE_HYPERPARAMETERS["lora_alpha"]
    WEIGHT_DECAY = BASE_HYPERPARAMETERS["weight_decay"]
    WARMUP_RATIO = BASE_HYPERPARAMETERS["warmup_ratio"]
    print("Using screened hyperparameters:", BASE_HYPERPARAMETERS)

In [ ]:
import shutil


def restore_run_folder(source, destination):
    """Copy attached read-only artifacts, relocating saved best-checkpoint paths."""
    source, destination = Path(source), Path(destination)
    if not (source / "run_config.json").is_file():
        raise FileNotFoundError("RESUME_SOURCE_DIR must be the run folder containing run_config.json.")
    if destination.exists() and any(destination.iterdir()):
        raise FileExistsError("Resume destination is not empty; restore into a clean working directory.")
    shutil.copytree(source, destination, dirs_exist_ok=True)
    # Colab and imported outputs can have different absolute root paths.
    # Work only on the writable copy; never mutate the attached input.
    for state_path in (destination / "checkpoints").rglob("trainer_state.json"):
        state = json.loads(state_path.read_text())
        best_path = state.get("best_model_checkpoint")
        if best_path:
            relocated = destination / "checkpoints" / Path(best_path).name
            if not relocated.is_dir():
                raise FileNotFoundError(f"Saved best checkpoint is missing: {relocated.name}. Attach the complete run folder.")
            state["best_model_checkpoint"] = str(relocated)
            state_path.write_text(json.dumps(state, indent=4) + "\n")


RUN_DIR = WORK_ROOT / RUN_NAME
CHECKPOINT_DIR = RUN_DIR / "checkpoints"
ADAPTER_DIR = RUN_DIR / "adapter"
INPUT_DIR = RUN_DIR / "input"
if RESUME and not (RUN_DIR / "run_config.json").is_file():
    if not RESUME_SOURCE_DIR:
        raise FileNotFoundError("A fresh background worker has no previous working files. Attach the saved run output and set RESUME_SOURCE_DIR.")
    restore_run_folder(RESUME_SOURCE_DIR, RUN_DIR)
for directory in (RUN_DIR, CHECKPOINT_DIR, INPUT_DIR):
    directory.mkdir(parents=True, exist_ok=True)
if (RUN_DIR / "run_config.json").exists() and not RESUME:
    raise FileExistsError("This run already exists. Use a new RUN_NAME or explicitly resume the original run.")


def save_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    clean_payload = json.loads(json.dumps(payload), parse_constant=lambda value: None)
    path.write_text(json.dumps(clean_payload, indent=4, ensure_ascii=False, allow_nan=False) + "\n", encoding="utf-8")


print(f"Run output: {RUN_DIR}")
print("Kaggle saves working files as notebook-version output. Verify the committed job finishes and its Output files are available.")

## 3. Read the attached training CSV

Attach `train_tickets.csv` using **Add Input**. With `DATA_CSV=""`, the notebook searches `/kaggle/input` and requires exactly one file with that name. If multiple datasets include one, set `DATA_CSV` to the full path shown by Kaggle. Resume can reuse the input CSV copied into the run folder.

The CSV, hash, and split IDs are saved in the run output. Validation rejects the held-out tenant, malformed targets, duplicates, and missing groups. Credentials should be stored in Kaggle Secrets, never in the CSV or notebook source.

In [ ]:
def find_training_csv(explicit_path, cached_path, input_root, search_result_path=""):
    if explicit_path:
        path = Path(explicit_path)
    elif Path(cached_path).is_file():
        path = Path(cached_path)
    elif search_result_path and (Path(search_result_path).parent / "input/train_tickets.csv").is_file():
        # Search outputs already contain their exact CSV; no second upload is needed.
        path = Path(search_result_path).parent / "input/train_tickets.csv"
    else:
        matches = sorted(Path(input_root).rglob("train_tickets.csv"))
        if len(matches) != 1:
            raise ValueError(f"Expected exactly one attached train_tickets.csv, found {len(matches)}. Set DATA_CSV to its full path.")
        path = matches[0]
    if not path.is_file():
        raise FileNotFoundError(f"Training CSV not found: {path}")
    return path


csv_path = find_training_csv(DATA_CSV, INPUT_DIR / "train_tickets.csv", "/kaggle/input", BEST_PARAMS_FILE)
raw_df = pd.read_csv(csv_path)
if raw_df.empty:
    raise ValueError("Training CSV contains no tickets.")
content = csv_path.read_bytes()
DATA_SHA256 = hashlib.sha256(content).hexdigest()
cached_input = INPUT_DIR / "train_tickets.csv"
if csv_path.resolve() != cached_input.resolve():
    cached_input.write_bytes(content)
del content
print(f"Loaded {len(raw_df):,} tickets | SHA-256: {DATA_SHA256}")

if BEST_PARAMS_PAYLOAD is not None:
    if BEST_PARAMS_PAYLOAD.get("data_sha256") != DATA_SHA256:
        raise ValueError("Search and full run use different CSV contents. Use the same training dataset.")
    save_json(RUN_DIR / "promoted_search_selection.json", BEST_PARAMS_PAYLOAD)

In [ ]:

def as_boolean(value):
    normalized = str(value).strip().lower()
    if normalized in {"true", "1"}:
        return True
    if normalized in {"false", "0"}:
        return False
    raise ValueError(f"Invalid is_answerable value: {value!r}")


def parse_evidence(value):
    evidence = json.loads(value) if isinstance(value, str) else value
    if not isinstance(evidence, list):
        raise ValueError("evidence_json must contain a list.")
    for item in evidence:
        if not isinstance(item, dict) or not all(
            isinstance(item.get(key), str) and item[key].strip() for key in ("source", "quote")
        ):
            raise ValueError("Each evidence item must have a nonempty source and quote.")
    return evidence


def validate_tickets(frame, held_out_tenant, task_mode):
    required = {"ticket_id", "batch_key", "business", "customer_query", "answer", "action",
                "confidence", "is_answerable", "evidence_json", "messages_json"}
    if task_mode == "label_and_rationale":
        required.add("rationale")
    missing = required - set(frame.columns)
    if missing:
        raise ValueError(f"Missing columns: {sorted(missing)}")
    frame = frame.copy()
    text_columns = required - {"confidence", "is_answerable"}
    for column in text_columns:
        if frame[column].isna().any() or frame[column].astype(str).str.strip().eq("").any():
            raise ValueError(f"Missing/empty values in {column}.")
    if frame["business"].astype(str).str.strip().str.casefold().eq(held_out_tenant.strip().casefold()).any():
        raise ValueError("Held-out tenant detected. Upload train_tickets.csv, not the full cleaned dataset.")
    if frame["ticket_id"].duplicated().any():
        raise ValueError("Duplicate ticket IDs detected.")
    if frame["customer_query"].astype(str).str.strip().str.casefold().duplicated().any():
        raise ValueError("Duplicate customer queries detected; deduplicate before splitting.")
    frame["is_answerable"] = frame["is_answerable"].map(as_boolean)
    frame["confidence"] = pd.to_numeric(frame["confidence"], errors="raise")
    if not frame["confidence"].between(0, 1).all():
        raise ValueError("confidence must be finite and between 0 and 1.")
    if not frame["action"].isin(["answer", "escalate"]).all():
        raise ValueError("Unknown action label.")
    if (~frame["is_answerable"] & frame["action"].ne("escalate")).any():
        raise ValueError("Unanswerable tickets must escalate.")
    frame["evidence"] = frame["evidence_json"].map(parse_evidence)
    if (frame["is_answerable"] & frame["evidence"].map(len).eq(0)).any():
        raise ValueError("Answer-present tickets must contain grounding evidence.")
    if (~frame["is_answerable"] & frame["evidence"].map(len).gt(0)).any():
        raise ValueError("Provably-absent tickets should not contain answer evidence.")
    # v2: optional `context_json` is what the model is SHOWN; `evidence_json` stays the cited support.
    # Without it, the context is the evidence (the original behaviour).
    if "context_json" in frame.columns:
        frame["context"] = frame["context_json"].map(parse_evidence)
        if frame["context"].map(len).eq(0).any():
            raise ValueError("context_json must contain at least one passage.")
        for ticket_id, evidence, context in zip(frame["ticket_id"], frame["evidence"], frame["context"]):
            shown = [" ".join(item["quote"].split()).casefold() for item in context]
            for item in evidence:
                needle = " ".join(item["quote"].split()).casefold()
                if not any(needle in text for text in shown):
                    raise ValueError(f"Cited evidence is not shown in the context for {ticket_id}.")
    else:
        frame["context"] = frame["evidence"]
    return frame.reset_index(drop=True)


def split_tickets(frame, validation_fraction, seed):
    # Keep all sibling tickets from a generation batch together, within each tenant.
    if not 0 < validation_fraction < 1:
        raise ValueError("validation_fraction must be between 0 and 1.")
    validation_ids = set()
    rng = random.Random(seed)
    for tenant, tenant_frame in frame.groupby("business", sort=True):
        groups = sorted(tenant_frame["batch_key"].unique().tolist())
        if len(groups) < 2:
            raise ValueError(f"Tenant {tenant} needs at least two batches to split safely.")
        rng.shuffle(groups)
        count = min(len(groups) - 1, max(1, math.ceil(len(groups) * validation_fraction)))
        selected = tenant_frame[tenant_frame["batch_key"].isin(groups[:count])]
        validation_ids.update(selected["ticket_id"])
    is_validation = frame["ticket_id"].isin(validation_ids)
    train, validation = frame[~is_validation].copy(), frame[is_validation].copy()
    train_groups = set(zip(train["business"], train["batch_key"]))
    validation_groups = set(zip(validation["business"], validation["batch_key"]))
    if train_groups & validation_groups:
        raise AssertionError("A generated batch leaked across splits.")
    if set(train["ticket_id"]) & set(validation["ticket_id"]):
        raise AssertionError("A ticket leaked across splits.")
    for name, part in (("training", train), ("validation", validation)):
        if set(part["action"]) != {"answer", "escalate"}:
            raise ValueError(f"The {name} split needs both answer and escalate targets.")
    return train.reset_index(drop=True), validation.reset_index(drop=True)

In [ ]:
tickets = validate_tickets(raw_df, HELD_OUT_TENANT, TASK_MODE)
train_df, validation_df = split_tickets(tickets, VALIDATION_FRACTION, SEED)
# Subsample only after splitting; every search trial reuses these same ticket IDs.
if RUN_MODE == "search":
    train_df = stratified_ticket_subset(train_df, SEARCH_TRAIN_TICKETS, SEED)
    validation_df = stratified_ticket_subset(validation_df, SEARCH_VALIDATION_TICKETS, SEED + 1)
elif RUN_MODE == "smoke":
    train_df = train_df.sample(n=min(SMOKE_MAX_TICKETS, len(train_df)), random_state=SEED).reset_index(drop=True)
    validation_df = validation_df.sample(n=min(32, len(validation_df)), random_state=SEED).reset_index(drop=True)

split_manifest = {
    "held_out_tenant": HELD_OUT_TENANT,
    "data_sha256": DATA_SHA256,
    "seed": SEED,
    "train_ticket_ids": sorted(train_df["ticket_id"].tolist()),
    "validation_ticket_ids": sorted(validation_df["ticket_id"].tolist()),
}
if RESUME:
    previous_split = json.loads((RUN_DIR / "split_manifest.json").read_text())
    if previous_split != split_manifest:
        raise ValueError("Input/split changed since this run. Restore the original CSV/settings or choose a new RUN_NAME.")
else:
    save_json(RUN_DIR / "split_manifest.json", split_manifest)

print(f"Training tickets: {len(train_df):,} | Validation tickets: {len(validation_df):,}")
display(tickets.groupby(["business", "action", "is_answerable"]).size().rename("tickets").to_frame())

## 4. Build grounded label and rationale tasks

Each ticket is split **before** task expansion. A JSON completion contains exactly `answer`, `cited_sources`, `confidence`, and `action`, matching `SupportResponse`. Teacher rationale is a separate completion; it is never inserted into the label prompt or mixed into the JSON answer.

Ground-truth evidence is grouped into source-tagged context blocks. Earlier customer/agent turns are included as conversation history, with the latest customer turn used as the question. Any agent response after that turn is excluded so the target answer cannot leak into the prompt. These are isolated generation-training examples; the held-out benchmark and live retrieval remain separate evaluation tools.

In [ ]:
# Mirrors src/inference/client.py; the standalone notebook does not need a repo checkout.
SUPPORT_SYSTEM_PROMPT = """You are a customer support assistant for a small business.
Answer the customer's question using ONLY the provided context documents.

Rules:
- If the context contains the answer, set "action" to "answer" and list the exact document names you used in "cited_sources".
- If the context does not contain the answer, or the request needs a human (emergencies, disputes, account changes), set "action" to "escalate", explain briefly in "answer", and set "cited_sources" to [].
- "confidence" is a number between 0.0 and 1.0 reflecting how fully the context supports your answer.
- Never invent policies, prices, or document names."""
TASK_SYSTEM_PROMPT = SUPPORT_SYSTEM_PROMPT + """
Return only a JSON object with answer, cited_sources, confidence, and action by default or when the question starts with [label].
When the question starts with [rationale], provide a short explanation of the decision grounded in the context; do not return the JSON answer."""


def format_evidence(evidence):
    grouped = {}
    for item in evidence:
        grouped.setdefault(item["source"], []).append(item["quote"].strip())
    return "\n\n".join(
        f'<context source="{escape(source, quote=True)}">\n[Source: {source}]\n'
        + "\n".join(quotes) + "\n</context>"
        for source, quotes in grouped.items()
    )


def ticket_user_prompt(row):
    messages = json.loads(row["messages_json"])
    if not isinstance(messages, list) or not messages:
        raise ValueError(f"Missing conversation for {row['ticket_id']}")
    for message in messages:
        if not isinstance(message, dict) or message.get("role") not in {"customer", "agent"}:
            raise ValueError(f"Invalid conversation role for {row['ticket_id']}")
        if not isinstance(message.get("content"), str) or not message["content"].strip():
            raise ValueError(f"Empty conversation turn for {row['ticket_id']}")
    customer_indices = [index for index, turn in enumerate(messages) if turn["role"] == "customer"]
    if not customer_indices:
        raise ValueError(f"No customer message for {row['ticket_id']}")
    last_customer = customer_indices[-1]
    if messages[last_customer]["content"].strip() != row["customer_query"].strip():
        raise ValueError(f"Conversation/query mismatch for {row['ticket_id']}")
    # Some cleaned conversations already include the final teacher response.
    # Exclude it: only history BEFORE the last customer question is input.
    messages = messages[:last_customer + 1]
    # v2: show the retrieved context when the CSV provides it; otherwise show the cited evidence.
    context = format_evidence(row.get("context") or row["evidence"])
    history = "\n".join(
        f"{'Customer' if turn['role'] == 'customer' else 'Support'}: {turn['content']}"
        for turn in messages[:-1]
    )
    history_block = f"\n\nConversation history:\n{history}" if history else ""
    return (f"Context documents:\n{context or '(no relevant documents found)'}"
            f"{history_block}\n\nCustomer question:\n{row['customer_query']}")


def build_task_examples(frame, task_mode, include_serving_labels):
    examples = []
    for row in frame.to_dict(orient="records"):
        user_prompt = ticket_user_prompt(row)
        sources = list(dict.fromkeys(item["source"] for item in row["evidence"]))
        target = json.dumps({
            "answer": row["answer"],
            "cited_sources": sources if row["action"] == "answer" else [],
            "confidence": float(row["confidence"]),
            "action": row["action"],
        }, ensure_ascii=False)
        tasks = [("label", "[label]\n" + user_prompt, target, TASK_SYSTEM_PROMPT)]
        if include_serving_labels:
            # Same unprefixed system/user prompt used by the existing inference client.
            tasks.append(("serving_label", user_prompt, target, SUPPORT_SYSTEM_PROMPT))
        if task_mode == "label_and_rationale":
            tasks.append(("rationale", "[rationale]\n" + user_prompt, row["rationale"], TASK_SYSTEM_PROMPT))
        for task, prompt, completion, system in tasks:
            examples.append({
                "ticket_id": row["ticket_id"], "task": task,
                "messages": [
                    {"role": "system", "content": system},
                    {"role": "user", "content": prompt},
                    {"role": "assistant", "content": completion},
                ],
            })
    return examples


train_examples = build_task_examples(train_df, TASK_MODE, INCLUDE_SERVING_LABELS)
validation_examples = build_task_examples(validation_df, TASK_MODE, INCLUDE_SERVING_LABELS)
print(f"Expanded examples: {len(train_examples):,} train / {len(validation_examples):,} validation")
print("First label target:", train_examples[0]["messages"][-1]["content"])

## 5. Load the 4-bit model and attach LoRA

The base weights stay quantized and frozen. LoRA trains only the adapter matrices on attention and feed-forward projections. Gradient checkpointing and a small batch size reduce GPU memory use; gradient accumulation gives an effective batch of eight examples on one GPU with the defaults.

In [ ]:
def load_qlora_model(params):
    set_seed(SEED)  # Reset adapter initialization and data randomness for each trial.
    fresh_model, fresh_tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_NAME, max_seq_length=MAX_SEQ_LENGTH,
        dtype=None, load_in_4bit=True, token=hf_token,
    )
    fresh_tokenizer = get_chat_template(fresh_tokenizer, chat_template="llama-3.1")
    fresh_tokenizer.padding_side = "right"
    if fresh_tokenizer.pad_token_id is None:
        fresh_tokenizer.pad_token = fresh_tokenizer.eos_token
    fresh_model = FastLanguageModel.get_peft_model(
        fresh_model, r=params["lora_rank"], lora_alpha=params["lora_alpha"],
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_dropout=0, bias="none", use_gradient_checkpointing="unsloth",
        random_state=SEED, use_rslora=False, loftq_config=None,
    )
    fresh_model.config.use_cache = False
    fresh_model.print_trainable_parameters()
    trainable, total = fresh_model.get_nb_trainable_parameters()
    if not 0 < trainable < total:
        raise RuntimeError("Expected trainable adapters on a frozen base model.")
    return fresh_model, fresh_tokenizer


if RUN_MODE == "search":
    # Prepare tokenization without holding a trial's GPU model between trials.
    from transformers import AutoTokenizer
    tokenizer = get_chat_template(AutoTokenizer.from_pretrained(MODEL_NAME, token=hf_token), chat_template="llama-3.1")
    tokenizer.padding_side = "right"
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
else:
    model, tokenizer = load_qlora_model(BASE_HYPERPARAMETERS)

## 6. Tokenize and verify completion masking

Only the assistant's completion contributes to loss. Prompt tokens (system instructions, context, conversation history, and the question) are masked with `-100`.

The length check stops rather than silently truncating answers or JSON. If it finds overlong examples, increase `MAX_SEQ_LENGTH` within your GPU budget or shorten the input upstream and start a new run. Packing is disabled so sample boundaries and completion masks stay easy to inspect.

In [ ]:
def tokenize_examples(examples, tokenizer, max_seq_length):
    encoded, overlong = [], []
    for example in examples:
        messages = example["messages"]
        # Apply the exact same chat template to the prompt and full conversation.
        prompt_ids = tokenizer.apply_chat_template(messages[:-1], tokenize=True, add_generation_prompt=True)
        full_ids = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=False)
        if full_ids[:len(prompt_ids)] != prompt_ids:
            raise ValueError("Chat-template prompt is not a prefix of the full conversation.")
        if len(full_ids) > max_seq_length:
            overlong.append((example["ticket_id"], example["task"], len(full_ids)))
            continue
        if len(full_ids) <= len(prompt_ids):
            raise ValueError(f"No completion tokens for {example['ticket_id']}")
        encoded.append({
            "input_ids": full_ids,
            "attention_mask": [1] * len(full_ids),
            "labels": [-100] * len(prompt_ids) + full_ids[len(prompt_ids):],
            "ticket_id": example["ticket_id"],
            "task": example["task"],
        })
    if overlong:
        raise ValueError(f"{len(overlong)} examples exceed MAX_SEQ_LENGTH={max_seq_length}; first cases: {overlong[:5]}")
    if not encoded:
        raise ValueError("No training examples were produced.")
    return encoded


# Precomputed labels avoid TRL-version-specific conversational masking behavior.
train_encoded = tokenize_examples(train_examples, tokenizer, MAX_SEQ_LENGTH)
validation_encoded = tokenize_examples(validation_examples, tokenizer, MAX_SEQ_LENGTH)
for name, encoded in (("train", train_encoded), ("validation", validation_encoded)):
    lengths = [len(item["input_ids"]) for item in encoded]
    print(f"{name}: min/median/max tokens = {min(lengths)}/{np.median(lengths):.0f}/{max(lengths)}")

train_dataset = Dataset.from_list([{key: row[key] for key in ("input_ids", "attention_mask", "labels")} for row in train_encoded])
validation_dataset = Dataset.from_list([{key: row[key] for key in ("input_ids", "attention_mask", "labels")} for row in validation_encoded])
# Explicit seq2seq collation preserves supplied -100 masks and pads labels with -100.
from transformers import DataCollatorForSeq2Seq
collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True, label_pad_token_id=-100, pad_to_multiple_of=8)
sample_batch = collator([train_dataset[index] for index in range(min(2, len(train_dataset)))])
assert (sample_batch["labels"] != -100).sum(dim=1).gt(0).all(), "A sample has no supervised tokens."
for index in range(sample_batch["labels"].shape[0]):
    labels = sample_batch["labels"][index]
    assert labels[0].item() == -100
    assert (labels[sample_batch["attention_mask"][index] == 0] == -100).all()
print("Supervised completion preview:")
print(tokenizer.decode(sample_batch["labels"][0][sample_batch["labels"][0] != -100].tolist()))
del sample_batch

## 7. Configure training and resumable output checkpoints

The training loop periodically evaluates and saves full optimizer/scheduler/RNG state. A callback verifies each saved checkpoint's files. Best-model selection uses validation loss, and old checkpoints are rotated to control output size.

A background Kaggle version is a fresh session; it cannot continue the Python objects from an editor session. Resume must reconstruct the trainer and load attached checkpoint files. Keep the original dataset hash and hyperparameters. Do not stop a save in progress. A notebook-version output is the durable handoff; files in a terminated unsaved interactive worker are not guaranteed recoverable.

In [ ]:
def verify_checkpoint(path):
    path = Path(path)
    required = ["trainer_state.json", "adapter_config.json", "optimizer.pt", "scheduler.pt"]
    missing = [name for name in required if not (path / name).is_file() or (path / name).stat().st_size == 0]
    adapter_files = [path / "adapter_model.safetensors", path / "adapter_model.bin"]
    if not any(file.is_file() and file.stat().st_size > 0 for file in adapter_files):
        missing.append("adapter_model.safetensors (or .bin)")
    if not any(file.is_file() and file.stat().st_size > 0 for file in path.glob("rng_state*.pth")):
        missing.append("rng_state.pth")
    if missing:
        raise RuntimeError(f"Incomplete checkpoint {path}: {missing}")
    state = json.loads((path / "trainer_state.json").read_text())
    if state.get("global_step", 0) <= 0:
        raise RuntimeError(f"Checkpoint has no completed optimizer steps: {path}")
    return {"path": str(path), "global_step": state["global_step"],
            "epoch": state.get("epoch"), "best_metric": state.get("best_metric")}


class CheckpointAuditCallback(TrainerCallback):
    def on_save(self, args, state, control, **kwargs):
        checkpoint = Path(args.output_dir) / f"checkpoint-{state.global_step}"
        report = verify_checkpoint(checkpoint)
        save_json(checkpoint / "checkpoint_verified.json", report)
        print(f"Verified persistent checkpoint at step {state.global_step}: {checkpoint}")
        return control

## Run the bounded hyperparameter search

Only search mode executes the next cell. It uses short, equal training budgets, ranks successful trials on a shared JSON-task validation subset, writes results after each trial, and releases the model/optimizer GPU allocations between trials. Failed trials stay visible and cannot win. Budgeted trials reduce both training and validation work; merely selecting fewer tickets would not reduce the number of updates if `SEARCH_MAX_STEPS` stayed unchanged.

Search mode skips the subsequent single-run training, generation, and GGUF cells. Full mode skips the search loop and follows the regular checkpointed training workflow below.

In [ ]:
import gc


def search_metadata():
    return {
        "model_name": MODEL_NAME, "data_sha256": DATA_SHA256,
        "held_out_tenant": HELD_OUT_TENANT, "seed": SEED,
        "validation_fraction": VALIDATION_FRACTION, "task_mode": TASK_MODE,
        "include_serving_labels": INCLUDE_SERVING_LABELS, "max_seq_length": MAX_SEQ_LENGTH,
    }


def search_loss_plot(history, baseline_loss, directory, title):
    frame = pd.DataFrame(history)
    frame.to_csv(directory / "loss_history.csv", index=False)
    fig, ax = plt.subplots(figsize=(8, 4))
    if "loss" in frame:
        rows = frame.dropna(subset=["loss"])
        ax.plot(rows["step"], rows["loss"], alpha=0.6, label="Training loss (task mix)")
    if "eval_loss" in frame:
        rows = frame.dropna(subset=["eval_loss"])
        ax.plot(rows["step"], rows["eval_loss"], marker="o", label="JSON-task validation loss")
    ax.axhline(baseline_loss, linestyle="--", color="gray", label="Before-training JSON-task loss")
    ax.set(title=title, xlabel="Optimizer step", ylabel="Completion cross-entropy")
    ax.legend()
    fig.tight_layout()
    fig.savefig(directory / "loss_curves.png", dpi=140)
    plt.close(fig)


def run_search_trial(spec, directory, scoring_dataset):
    # No model/trainer escapes this function: each trial starts from the original base.
    trial_model = trial_tokenizer = trial_trainer = None
    started = time.perf_counter()
    params = spec["hyperparameters"]
    directory.mkdir(parents=True, exist_ok=False)
    save_json(directory / "hyperparameters.json", params)
    torch.cuda.reset_peak_memory_stats()
    try:
        trial_model, trial_tokenizer = load_qlora_model(params)
        if (trial_tokenizer.chat_template != tokenizer.chat_template
                or trial_tokenizer.pad_token_id != tokenizer.pad_token_id
                or trial_tokenizer.eos_token_id != tokenizer.eos_token_id):
            raise RuntimeError("Trial tokenizer differs from the shared pretokenized dataset.")
        args = SFTConfig(
            output_dir=str(directory / "checkpoints"), run_name=spec["trial_id"],
            per_device_train_batch_size=TRAIN_BATCH_SIZE, per_device_eval_batch_size=1,
            gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
            learning_rate=params["learning_rate"], optim=params["optimizer"],
            weight_decay=params["weight_decay"], warmup_ratio=params["warmup_ratio"],
            lr_scheduler_type="linear", max_steps=SEARCH_MAX_STEPS,
            fp16=not is_bfloat16_supported(), bf16=is_bfloat16_supported(),
            logging_steps=10, logging_first_step=True, logging_nan_inf_filter=False,
            eval_strategy="steps", eval_steps=SEARCH_EVAL_STEPS,
            save_strategy="steps", save_steps=SEARCH_EVAL_STEPS, save_total_limit=1,
            save_only_model=False, load_best_model_at_end=True,
            metric_for_best_model="eval_loss", greater_is_better=False,
            prediction_loss_only=True, eval_accumulation_steps=1,
            gradient_checkpointing=True, max_grad_norm=1.0,
            seed=SEED, data_seed=SEED, dataloader_num_workers=0,
            report_to="none", push_to_hub=False, max_length=MAX_SEQ_LENGTH,
            packing=False, dataset_kwargs={"skip_prepare_dataset": True},
        )
        trial_trainer = SFTTrainer(
            model=trial_model, processing_class=trial_tokenizer, args=args,
            train_dataset=train_dataset, eval_dataset=scoring_dataset,
            data_collator=collator, callbacks=[CheckpointAuditCallback()],
        )
        baseline = trial_trainer.evaluate(metric_key_prefix="baseline")
        save_json(directory / "baseline_validation.json", baseline)
        trained = trial_trainer.train()
        selected = trial_trainer.evaluate(metric_key_prefix="selected")
        loss = float(selected["selected_loss"])
        loss_history = trial_trainer.state.log_history
        observed_losses = [float(baseline["baseline_loss"]), loss]
        observed_losses.extend(float(row[key]) for row in loss_history for key in ("loss", "eval_loss") if key in row)
        if not all(math.isfinite(value) for value in observed_losses):
            raise RuntimeError("Nonfinite loss: this trial cannot be selected.")
        adapter_dir = directory / "adapter"
        trial_trainer.save_model(str(adapter_dir))
        trial_tokenizer.save_pretrained(str(adapter_dir))
        trial_trainer.save_state()
        reports = [verify_checkpoint(path) for path in (directory / "checkpoints").glob("checkpoint-*")]
        if not reports:
            raise RuntimeError("Trial finished without a resumable checkpoint.")
        save_json(directory / "checkpoint_audit.json", reports)
        save_json(directory / "training_log_history.json", loss_history)
        search_loss_plot(loss_history, baseline["baseline_loss"], directory, spec["trial_id"])
        result = {
            **spec, "status": "completed", "validation_loss": loss,
            "baseline_validation_loss": float(baseline["baseline_loss"]),
            "optimizer_steps": trial_trainer.state.global_step,
            "best_checkpoint": trial_trainer.state.best_model_checkpoint,
            "adapter_dir": str(adapter_dir), "training_metrics": trained.metrics,
            "selected_metrics": selected, "directory": str(directory),
        }
    except KeyboardInterrupt:
        save_json(directory / "interrupted.json", {"status": "interrupted", "trial_id": spec["trial_id"]})
        raise
    except Exception as exc:
        result = {**spec, "status": "failed", "validation_loss": None,
                  "directory": str(directory), "error": f"{type(exc).__name__}: {str(exc)[:2000]}"}
    finally:
        if trial_trainer is not None:
            save_json(directory / "log_history.json", trial_trainer.state.log_history)
            trial_trainer.accelerator.free_memory()
        trial_trainer = trial_model = trial_tokenizer = None
        gc.collect()
        torch.cuda.empty_cache()
    result["wall_seconds"] = time.perf_counter() - started
    save_json(directory / "result.json", result)
    return result


def persist_search_results(results, metadata, planned_trials, complete):
    ranked = rank_search_results(results)
    rank_map = {row["trial_id"]: index for index, row in enumerate(ranked, 1)}
    rows = [{
        "rank": rank_map.get(row["trial_id"]), "trial_id": row["trial_id"],
        "status": row["status"], "validation_loss": row.get("validation_loss"),
        "wall_seconds": row.get("wall_seconds"), **row["hyperparameters"],
        "error": row.get("error", ""),
    } for row in results]
    table = pd.DataFrame(rows)
    if not table.empty:
        table = table.sort_values(["rank", "trial_id"], na_position="last")
    table.to_csv(RUN_DIR / "search_results.csv", index=False)
    save_json(RUN_DIR / "search_results.json", {
        **metadata, "search_complete": complete, "planned_trials": planned_trials,
        "successful_trials": len(ranked), "results": results,
    })
    if ranked:
        best = ranked[0]
        save_json(RUN_DIR / "best_hyperparameters.json", {
            **metadata, "search_complete": complete,
            "planned_trials": planned_trials, "successful_trials": len(ranked),
            "best_trial_id": best["trial_id"], "validation_loss": best["validation_loss"],
            "hyperparameters": best["hyperparameters"],
        })
    return table


if RUN_MODE == "search":
    plan, combinations = build_search_plan(SEARCH_SPACE, BASE_HYPERPARAMETERS, SEARCH_METHOD, SEARCH_MAX_TRIALS, SEED)
    scoring_task = "serving_label" if INCLUDE_SERVING_LABELS else "label"
    scoring_indices = [index for index, item in enumerate(validation_encoded) if item["task"] == scoring_task]
    scoring_dataset = validation_dataset.select(scoring_indices)
    if len(scoring_dataset) == 0:
        raise ValueError("Search has no JSON-task validation examples.")
    metadata = {**search_metadata(), "objective": f"{scoring_task}_completion_loss"}
    search_config = {
        **metadata, "run_mode": "search", "plan": plan, "combinations": combinations,
        "train_batch_size": TRAIN_BATCH_SIZE, "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
        "train_tickets": len(train_df), "validation_tickets": len(validation_df),
        "train_examples": len(train_dataset), "max_steps_per_trial": SEARCH_MAX_STEPS,
        "eval_steps": SEARCH_EVAL_STEPS,
    }
    config_path = RUN_DIR / "run_config.json"
    versions = {name: importlib.metadata.version(name) for name in (
        "unsloth", "unsloth_zoo", "torch", "transformers", "trl", "datasets", "peft", "accelerate", "bitsandbytes"
    )}
    if RESUME:
        if json.loads(config_path.read_text()) != search_config:
            raise ValueError("Search settings changed. Use the original settings to resume or a new RUN_NAME.")
        previous_versions = json.loads((RUN_DIR / "environment.json").read_text())["packages"]
        if previous_versions != versions and not ALLOW_ENVIRONMENT_CHANGE:
            raise RuntimeError("Search package versions changed. Restore them or explicitly acknowledge environment migration.")
    else:
        save_json(config_path, search_config)
        save_json(RUN_DIR / "environment.json", {"python": sys.version, "gpu": torch.cuda.get_device_name(0), "packages": versions})
        (RUN_DIR / "requirements-lock.txt").write_text(
            "\n".join(sorted(f"{d.metadata['Name']}=={d.version}" for d in importlib.metadata.distributions() if d.metadata['Name'])) + "\n"
        )
    save_json(RUN_DIR / "search_plan.json", search_config)
    print(f"{len(plan)} selected trials out of {combinations} unique combinations.")
    print(f"Budget: {len(plan) * SEARCH_MAX_STEPS} optimizer steps; score on {len(scoring_dataset)} fixed {scoring_task} examples.")
    if SEARCH_METHOD == "grid" and len(plan) < combinations:
        print("Grid is capped: only the first combinations will run. Raise SEARCH_MAX_TRIALS for all, or use random to spread the budget.")
    results = []
    for spec in plan:
        trial_root = RUN_DIR / "trials" / spec["trial_id"]
        prior_path = trial_root / "result.json"
        prior = json.loads(prior_path.read_text()) if prior_path.is_file() else None
        if RESUME and prior and rank_search_results([prior]) and prior["hyperparameters"] == spec["hyperparameters"]:
            print(f"Reusing completed {spec['trial_id']}")
            results.append(prior)
        else:
            trial_root.mkdir(parents=True, exist_ok=True)
            attempt = 1
            while (trial_root / f"attempt_{attempt:02d}").exists():
                attempt += 1
            print(f"Starting {spec['trial_id']}: {spec['hyperparameters']}")
            result = run_search_trial(spec, trial_root / f"attempt_{attempt:02d}", scoring_dataset)
            save_json(prior_path, result)
            results.append(result)
            print(f"{spec['trial_id']}: {result['status']} | validation loss: {result.get('validation_loss')}")
        persist_search_results(results, metadata, len(plan), complete=False)
    leaderboard = persist_search_results(results, metadata, len(plan), complete=True)
    display(leaderboard)
    if not rank_search_results(results):
        raise RuntimeError("All search trials failed. Inspect search_results.json and per-trial error logs.")
    print(f"Search complete: {RUN_DIR / 'best_hyperparameters.json'}")
    print("Next: attach this saved output to a new full run and set BEST_PARAMS_FILE='auto'. Full training starts from a fresh base model.")

In [ ]:
if RUN_MODE != "search":
    EVAL_SAVE_STEPS = 10 if RUN_MODE == "smoke" else FULL_EVAL_SAVE_STEPS
    MAX_STEPS = SMOKE_MAX_STEPS if RUN_MODE == "smoke" else -1
    RESUME_CHECKPOINT = get_last_checkpoint(str(CHECKPOINT_DIR)) if RESUME else None
    if RESUME:
        if RESUME_CHECKPOINT is None:
            raise FileNotFoundError("No checkpoint found for this run. Start a new RUN_NAME with RESUME=False if training never reached a save.")
        print("Resuming:", verify_checkpoint(RESUME_CHECKPOINT))

    run_config = {
        "model_name": MODEL_NAME, "run_name": RUN_NAME, "run_mode": RUN_MODE,
        "data_sha256": DATA_SHA256, "held_out_tenant": HELD_OUT_TENANT,
        "seed": SEED, "validation_fraction": VALIDATION_FRACTION,
        "task_mode": TASK_MODE, "include_serving_labels": INCLUDE_SERVING_LABELS,
        "max_seq_length": MAX_SEQ_LENGTH, "lora_rank": LORA_RANK, "lora_alpha": LORA_ALPHA,
        "learning_rate": LEARNING_RATE, "num_train_epochs": NUM_TRAIN_EPOCHS,
        "optimizer": OPTIMIZER, "weight_decay": WEIGHT_DECAY, "warmup_ratio": WARMUP_RATIO,
        "train_batch_size": TRAIN_BATCH_SIZE, "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
        "eval_save_steps": EVAL_SAVE_STEPS, "max_steps": MAX_STEPS,
        "train_examples": len(train_dataset), "validation_examples": len(validation_dataset),
    }
    if RESUME:
        previous_config = json.loads((RUN_DIR / "run_config.json").read_text())
        # Defaults preserve compatibility with earlier single-run notebook manifests.
        for key, default in {"optimizer": "adamw_8bit", "weight_decay": 0.01, "warmup_ratio": 0.03}.items():
            previous_config.setdefault(key, default)
        if previous_config != run_config:
            raise ValueError("Run configuration changed. Restore the original settings for an exact resume.")
    else:
        save_json(RUN_DIR / "run_config.json", run_config)

    package_versions = {name: importlib.metadata.version(name) for name in (
        "unsloth", "unsloth_zoo", "torch", "transformers", "trl", "datasets", "peft", "accelerate", "bitsandbytes"
    )}
    save_json(RUN_DIR / ("resume_environment.json" if RESUME else "environment.json"), {
        "python": sys.version, "gpu": torch.cuda.get_device_name(0),
        "cuda": torch.version.cuda, "packages": package_versions,
    })
    if RESUME:
        original_packages = json.loads((RUN_DIR / "environment.json").read_text())["packages"]
        if original_packages != package_versions:
            if not ALLOW_ENVIRONMENT_CHANGE:
                raise RuntimeError("Package versions differ from the saved run. Match its versions, or explicitly set ALLOW_ENVIRONMENT_CHANGE=True for an intentional migration. Verify optimizer/checkpoint compatibility first.")
            print("Environment migration acknowledged. Training-state restoration will be attempted; identical numerical behavior is not guaranteed.")
            save_json(RUN_DIR / "environment_migration.json", {"original": original_packages, "current": package_versions})
    else:
        (RUN_DIR / "requirements-lock.txt").write_text(
            "\n".join(sorted(f"{distribution.metadata['Name']}=={distribution.version}" for distribution in importlib.metadata.distributions() if distribution.metadata['Name'])) + "\n"
        )
    if USE_WANDB:
        wandb_run = wandb.init(project=WANDB_PROJECT, name=RUN_NAME, id=RUN_NAME,
                               resume="allow", config=run_config)

    training_args = SFTConfig(
        output_dir=str(CHECKPOINT_DIR),
        run_name=RUN_NAME,
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        learning_rate=LEARNING_RATE,
        num_train_epochs=NUM_TRAIN_EPOCHS,
        max_steps=MAX_STEPS,
        warmup_ratio=WARMUP_RATIO,
        lr_scheduler_type="linear",
        optim=OPTIMIZER,
        weight_decay=WEIGHT_DECAY,
        max_grad_norm=1.0,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=1 if RUN_MODE == "smoke" else 10,
        logging_first_step=True,
        logging_nan_inf_filter=False,
        eval_strategy="steps",
        eval_steps=EVAL_SAVE_STEPS,
        save_strategy="steps",
        save_steps=EVAL_SAVE_STEPS,
        save_total_limit=2,
        save_only_model=False,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        prediction_loss_only=True,
        eval_accumulation_steps=1,
        gradient_checkpointing=True,
        seed=SEED,
        data_seed=SEED,
        dataloader_num_workers=0,
        report_to=["wandb"] if USE_WANDB else "none",
        push_to_hub=False,
        max_length=MAX_SEQ_LENGTH,
        packing=False,
        dataset_kwargs={"skip_prepare_dataset": True},
    )
    trainer = SFTTrainer(
        model=model,
        processing_class=tokenizer,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=validation_dataset,
        data_collator=collator,
        callbacks=[CheckpointAuditCallback()],
    )
    steps_per_epoch = math.ceil(len(train_dataset) / (TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS))
    planned_steps = MAX_STEPS if MAX_STEPS > 0 else math.ceil(steps_per_epoch * NUM_TRAIN_EPOCHS)
    if planned_steps < EVAL_SAVE_STEPS:
        raise ValueError("This run would finish before its first checkpoint. Reduce FULL_EVAL_SAVE_STEPS or increase training duration.")
    print(f"Effective batch size: {TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}")
    print(f"Planned optimizer steps: {planned_steps}; evaluate/save every {EVAL_SAVE_STEPS}")
    print("Package versions:", package_versions)

In [ ]:
if RUN_MODE != "search":
    # Save a validation-loss reference before adapter training. On resume, preserve it.
    baseline_path = RUN_DIR / "baseline_validation.json"
    if RESUME:
        if not baseline_path.is_file():
            raise FileNotFoundError("Missing baseline_validation.json for this run.")
        baseline_metrics = json.loads(baseline_path.read_text())
    else:
        baseline_metrics = trainer.evaluate(metric_key_prefix="baseline")
        save_json(baseline_path, baseline_metrics)
    print("Before-training validation loss:", baseline_metrics["baseline_loss"])

In [ ]:
if RUN_MODE != "search":
    # Changing RESUME requires rerunning configuration and trainer setup first.
    # Reconstruct the trainer in a fresh session for recovery, not an interrupted in-memory model.
    torch.cuda.reset_peak_memory_stats()
    started = time.perf_counter()
    try:
        training_result = trainer.train(resume_from_checkpoint=RESUME_CHECKPOINT)
    except (KeyboardInterrupt, Exception):
        # Retain evidence on a graceful interrupt/error; completed checkpoints remain resumable.
        save_json(RUN_DIR / "interrupted_log_history.json", trainer.state.log_history)
        save_json(RUN_DIR / "interruption.json", {
            "timestamp": datetime.now(timezone.utc).isoformat(),
            "global_step": trainer.state.global_step,
            "latest_checkpoint": get_last_checkpoint(str(CHECKPOINT_DIR)),
        })
        if wandb_run is not None:
            wandb.finish(exit_code=1)
        raise
    training_wall_seconds = time.perf_counter() - started
    trainer.save_state()
    trainer.save_metrics("train", training_result.metrics)
    save_json(RUN_DIR / "training_log_history.json", trainer.state.log_history)
    print(f"Training call finished in {training_wall_seconds / 60:.1f} minutes.")
    print(f"Peak allocated GPU memory: {torch.cuda.max_memory_allocated() / 2**30:.2f} GiB")

## 8. Verify loss behavior and inspect convergence

The baseline, step evaluations, and final selected-adapter evaluation use the same validation examples and completion masks. Loss is token-level cross-entropy on the selected task mix, **not** a hallucination or escalation score. The notebook also reports validation loss separately for each task.

Look for finite losses and a sustained validation-loss improvement, with training and validation curves moving in a compatible direction. Decreasing training loss with increasing validation loss suggests overfitting. A one-epoch run or a smoke test may not establish convergence; the report records observations and leaves the conclusion to the measured curves. The best checkpoint may occur before the final optimizer step.

In [ ]:
if RUN_MODE != "search":
    final_metrics = trainer.evaluate(metric_key_prefix="selected")
    trainer.save_metrics("selected_validation", final_metrics)
    task_metrics = {}
    for task in sorted({row["task"] for row in validation_encoded}):
        indices = [index for index, row in enumerate(validation_encoded) if row["task"] == task]
        task_metrics[task] = trainer.evaluate(validation_dataset.select(indices), metric_key_prefix=f"validation_{task}")
    save_json(RUN_DIR / "validation_by_task.json", task_metrics)

    history = pd.DataFrame(trainer.state.log_history)
    history.to_csv(RUN_DIR / "loss_history.csv", index=False)
    train_curve = history.dropna(subset=["loss"]) if "loss" in history else pd.DataFrame()
    eval_curve = history.dropna(subset=["eval_loss"]) if "eval_loss" in history else pd.DataFrame()
    baseline_loss = float(baseline_metrics["baseline_loss"])
    selected_loss = float(final_metrics["selected_loss"])
    loss_values = [baseline_loss, selected_loss]
    for column in ("loss", "eval_loss"):
        if column in history:
            loss_values.extend(history[column].dropna().astype(float).tolist())
    finite_losses = bool(np.isfinite(loss_values).all())
    summary = {
        "run_mode": RUN_MODE,
        "smoke_test_only": RUN_MODE == "smoke",
        "optimizer_steps": trainer.state.global_step,
        "training_wall_seconds_this_session": training_wall_seconds,
        "peak_gpu_memory_gib": torch.cuda.max_memory_allocated() / 2**30,
        "baseline_validation_loss": baseline_loss,
        "selected_validation_loss": selected_loss,
        "validation_loss_change": selected_loss - baseline_loss,
        "validation_loss_improved": bool(selected_loss < baseline_loss),
        "all_recorded_losses_finite": finite_losses,
        "evaluation_points": len(eval_curve),
        "best_checkpoint": trainer.state.best_model_checkpoint,
        "best_checkpoint_validation_loss": trainer.state.best_metric,
        "convergence_claim": "Review full-run curves; smoke runs verify plumbing only.",
        "training_metrics": training_result.metrics,
        "validation_metrics": final_metrics,
    }
    # Preserve diagnostics even if a run diverged (JSON cannot encode NaN/Infinity).
    summary_for_json = json.loads(json.dumps(summary), parse_constant=lambda value: None)
    save_json(RUN_DIR / "run_summary.json", summary_for_json)

    fig, ax = plt.subplots(figsize=(10, 5))
    if not train_curve.empty:
        ax.plot(train_curve["step"], train_curve["loss"], alpha=0.35, label="Training loss (raw)")
        ax.plot(train_curve["step"], train_curve["loss"].rolling(5, min_periods=1).mean(), label="Training loss (5-log mean)")
    ax.axhline(baseline_loss, color="gray", linestyle="--", label="Before-training validation loss")
    if not eval_curve.empty:
        ax.plot(eval_curve["step"], eval_curve["eval_loss"], marker="o", label="Validation loss")
    ax.set(xlabel="Optimizer step", ylabel="Completion cross-entropy loss", title=f"{RUN_NAME} ({RUN_MODE})")
    ax.grid(alpha=0.2)
    ax.legend()
    fig.tight_layout()
    fig.savefig(RUN_DIR / "loss_curves.png", dpi=160)
    fig.savefig(RUN_DIR / "loss_curves.svg")
    plt.show()
    display(pd.DataFrame([{
        "baseline_validation_loss": baseline_loss, "selected_validation_loss": selected_loss,
        "change": selected_loss - baseline_loss, "finite_losses": finite_losses,
        "best_step": Path(trainer.state.best_model_checkpoint).name if trainer.state.best_model_checkpoint else None,
    }]))
    if not finite_losses:
        raise RuntimeError("Nonfinite loss detected; inspect the logs before accepting this run.")
    print("Validation loss improved." if selected_loss < baseline_loss else "Validation loss did not improve; inspect the curves and run settings.")
    if RUN_MODE == "smoke":
        print("SMOKE TEST: do not report convergence from this short run. Run the full profile next.")

## 9. Save the selected adapter and verify checkpoint files

`adapter/` contains the adapter selected by validation loss plus its tokenizer/template. `checkpoints/checkpoint-*` contains the resumable training state. Keep both: the adapter is small and useful for inference/export, while the checkpoints allow training to resume.

The file audit checks nonempty adapter, optimizer, scheduler, RNG, and Trainer-state files. The optional fresh-session reload in the next section is a separate inference check. All artifacts remain in the saved Kaggle version output after the Kaggle worker exits and the version output has been saved.

In [ ]:
if RUN_MODE != "search":
    trainer.save_model(str(ADAPTER_DIR))
    tokenizer.save_pretrained(str(ADAPTER_DIR))
    if not (ADAPTER_DIR / "adapter_config.json").is_file():
        raise RuntimeError("Adapter configuration was not saved.")
    if not any((ADAPTER_DIR / filename).is_file() and (ADAPTER_DIR / filename).stat().st_size > 0
               for filename in ("adapter_model.safetensors", "adapter_model.bin")):
        raise RuntimeError("Adapter weights were not saved.")
    if not (ADAPTER_DIR / "tokenizer_config.json").is_file():
        raise RuntimeError("Tokenizer/chat-template configuration was not saved.")
    checkpoint_paths = sorted(CHECKPOINT_DIR.glob("checkpoint-*"), key=lambda path: int(path.name.rsplit("-", 1)[1]))
    if not checkpoint_paths:
        raise RuntimeError("No periodic checkpoint was saved.")
    checkpoint_reports = [verify_checkpoint(path) for path in checkpoint_paths]
    save_json(RUN_DIR / "checkpoint_audit.json", checkpoint_reports)
    display(pd.DataFrame(checkpoint_reports))
    print(f"Selected adapter/tokenizer: {ADAPTER_DIR}")
    print("Best checkpoint:", trainer.state.best_model_checkpoint)

## 10. Generate and inspect validation examples

Test the selected adapter on one answer, one provably-absent escalation, and (when available) one documented human escalation from the **validation** split. Label generations use the same unprefixed prompt as the existing client when `INCLUDE_SERVING_LABELS=True`. These generations are unconstrained, so parsing checks are useful; deployment still uses the client's schema-constrained decoding.

This small review is not the final held-out benchmark. After exporting and serving the model, use `scripts/evaluate_generation.py` on the frozen benchmark for hallucination and escalation metrics. Do not tune on that benchmark during these training runs.

In [ ]:
if RUN_MODE != "search":
    FastLanguageModel.for_inference(model)
    model.config.use_cache = True
    review_frames = [
        validation_df[validation_df["action"].eq("answer")].head(1),
        validation_df[~validation_df["is_answerable"]].head(1),
        validation_df[validation_df["is_answerable"] & validation_df["action"].eq("escalate")].head(1),
    ]
    review_df = pd.concat(review_frames).drop_duplicates(subset="ticket_id")
    review_records = []
    for row in review_df.to_dict(orient="records"):
        system_prompt = SUPPORT_SYSTEM_PROMPT if INCLUDE_SERVING_LABELS else TASK_SYSTEM_PROMPT
        user_prompt = ticket_user_prompt(row)
        if not INCLUDE_SERVING_LABELS:
            user_prompt = "[label]\n" + user_prompt
        prompt_ids = tokenizer.apply_chat_template(
            [{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}],
            tokenize=True, add_generation_prompt=True, return_tensors="pt",
        ).to("cuda")
        with torch.inference_mode():
            generated = model.generate(
                input_ids=prompt_ids, attention_mask=torch.ones_like(prompt_ids),
                max_new_tokens=512, do_sample=False, use_cache=True,
                pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id,
            )
        raw = tokenizer.decode(generated[0, prompt_ids.shape[-1]:], skip_special_tokens=True).strip()
        parsed, error = None, None
        try:
            candidate = json.loads(raw)
            if not isinstance(candidate, dict) or not {"answer", "cited_sources", "confidence", "action"}.issubset(candidate):
                raise ValueError("Missing response fields.")
            if candidate["action"] not in ("answer", "escalate") or not isinstance(candidate["answer"], str):
                raise ValueError("Invalid action or answer.")
            if type(candidate["confidence"]) not in (int, float) or not 0 <= candidate["confidence"] <= 1:
                raise ValueError("Invalid confidence.")
            if not isinstance(candidate["cited_sources"], list) or not all(isinstance(source, str) for source in candidate["cited_sources"]):
                raise ValueError("Invalid cited_sources.")
            parsed = candidate
        except (ValueError, TypeError) as exc:
            error = str(exc)
        review_records.append({
            "ticket_id": row["ticket_id"], "query": row["customer_query"],
            "expected_action": row["action"], "is_answerable": bool(row["is_answerable"]),
            "ground_truth_answer": row["answer"], "raw": raw, "parsed": parsed, "error": error,
            "action_correct": parsed is not None and parsed["action"] == row["action"],
        })
        print(f"\n{row['ticket_id']} | expected {row['action']} | parsed={parsed is not None}\n{raw}")
    save_json(RUN_DIR / "validation_generations.json", review_records)
    if wandb_run is not None:
        wandb.log({"selected_validation_loss": selected_loss, "baseline_validation_loss": baseline_loss})
        wandb.finish()

## 11. Optional adapter reload

This check is disabled for the normal background training run. To test persisted weights independently, attach the saved output in a fresh Kaggle session, run installation/configuration/imports/storage with `RESUME=True` and the original `RUN_NAME`, then enable this cell. Do not first load the training model. The exported adapter tests inference; optimizer-level continuation uses the full checkpoint workflow instead.

In [ ]:
if RUN_MODE != "search":
    RELOAD_SAVED_ADAPTER = False
    if RELOAD_SAVED_ADAPTER:
        if "model" in globals():
            raise RuntimeError("Use a fresh Kaggle session for reload verification, without running the training model cell.")
        reload_dir = WORK_ROOT / RUN_NAME / "adapter"
        reloaded_model, reloaded_tokenizer = FastLanguageModel.from_pretrained(
            model_name=str(reload_dir), max_seq_length=MAX_SEQ_LENGTH, dtype=None,
            load_in_4bit=True, token=hf_token,
        )
        FastLanguageModel.for_inference(reloaded_model)
        reload_messages = [
            {"role": "system", "content": "Answer using only the context. Return JSON with answer, cited_sources, confidence, action."},
            {"role": "user", "content": "Context documents:\n(no relevant documents found)\n\nCustomer question:\nWhat is your unpublished VIP discount?"},
        ]
        reload_ids = reloaded_tokenizer.apply_chat_template(
            reload_messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
        ).to("cuda")
        with torch.inference_mode():
            reload_output = reloaded_model.generate(
                input_ids=reload_ids, attention_mask=torch.ones_like(reload_ids),
                do_sample=False, max_new_tokens=256, pad_token_id=reloaded_tokenizer.pad_token_id,
            )
        print(reloaded_tokenizer.decode(reload_output[0, reload_ids.shape[-1]:], skip_special_tokens=True))
        print("Reload succeeded from saved adapter:", reload_dir)

## 12. Optional GGUF export for a later Ollama/llama.cpp evaluation

For a **fresh session after full training has finished**, use the separate `export_gguf_kaggle.ipynb` notebook with the completed full-run output attached. It initializes its own variables and reloads the saved adapter/tokenizer without training. The optional export cell in this training notebook depends on the earlier setup and an already loaded `model`/`tokenizer`; it is not standalone.

After verifying the saved checkpoints, curves, and adapter, set `EXPORT_GGUF=True` to export a deployment candidate. `GGUF_QUANTIZATIONS` defaults to `["q4_k_m"]`; choose `["q4_k_m", "q5_k_m", "q8_0"]` for the three-format comparison. Formats are exported sequentially from the same selected model; valid existing exports are reused. `gguf_export.json` records partial/completed formats, paths, and sizes. Header checks do not verify inference quality or peak memory. Export merges the adapter and builds/uses llama.cpp in the Kaggle worker; allow additional disk, RAM, runtime, and network access. It is a separate step and does not affect the saved adapter output if conversion fails.

Keep the saved Llama chat template when serving the result. Inspect an exported Modelfile's system prompt/template before importing into Ollama, and use the JSON support prompt for label inference. With the default serving-label examples, the existing inference client's unprefixed prompt is included in training.

Unsloth versions may write to a sibling `gguf_gguf` folder. The export cell recovers an existing Q4_K_M file there, checks its GGUF header, and moves it into this run's `gguf` folder without overwriting files. Rerunning the cell reuses that export; it does not repeat conversion. Use a new run name for new training. Header checks do not replace a local inference test.


In [ ]:
def validate_gguf_quantizations(quantizations):
    if not isinstance(quantizations, (list, tuple)) or not quantizations:
        raise ValueError("GGUF_QUANTIZATIONS must be a nonempty list of q4_k_m, q5_k_m, or q8_0.")
    methods = [str(value).lower() for value in quantizations]
    if len(set(methods)) != len(methods) or any(method not in {"q4_k_m", "q5_k_m", "q8_0"} for method in methods):
        raise ValueError("Choose unique GGUF quantizations from q4_k_m, q5_k_m, q8_0.")
    return methods


def collect_gguf_export(run_dir, quantization="q4_k_m"):
    """Recover only the requested quantization, validating all moves first.

    Other quantizations and intermediate F16 files stay untouched. A GGUF
    header/file-size check is not an inference or model-quality test.
    """
    import re
    import shutil

    method = validate_gguf_quantizations([quantization])[0]
    run_dir = Path(run_dir)
    destination = run_dir / "gguf"
    roots = (destination, run_dir / "gguf_gguf")
    candidates = []
    for root in roots:
        if not root.is_dir():
            continue
        for path in sorted(root.rglob("*.gguf")):
            if not re.search(r"(?:^|[._-])" + re.escape(method) + r"\.gguf$", path.name.lower()) or not path.is_file():
                continue
            if path.stat().st_size < 24:
                raise RuntimeError(f"GGUF export is too small: {path}")
            with path.open("rb") as handle:
                if handle.read(4) != b"GGUF":
                    raise RuntimeError(f"Invalid GGUF header: {path}")
            candidates.append(path)
    targets = {}
    for source in candidates:
        target = destination / source.name
        if target in targets or (target.exists() and source != target):
            raise FileExistsError(f"Conflicting GGUF exports for {target}; review both files before retrying.")
        targets[target] = source
    if candidates:
        destination.mkdir(parents=True, exist_ok=True)
    for target, source in targets.items():
        if source != target:
            shutil.move(str(source), str(target))
    return sorted(targets)


def export_gguf_variants(run_dir, quantizations, exporter, save_report):
    """Export/recover selected variants sequentially and persist partial progress.

    exporter is called only for missing formats; it must use the same selected
    full-run model and tokenizer for each format. Existing exports are reused.
    """
    methods = validate_gguf_quantizations(quantizations)
    run_dir = Path(run_dir)
    exports = {}
    for method in methods:
        files = collect_gguf_export(run_dir, method)
        if files:
            print(f"Reusing existing {method.upper()} export; conversion was not repeated.")
        else:
            exporter(method)
            files = collect_gguf_export(run_dir, method)
        if not files:
            raise RuntimeError(f"No {method.upper()} GGUF found in {run_dir / 'gguf'} or {run_dir / 'gguf_gguf'}. Inspect conversion output paths/logs.")
        exports[method] = [str(path) for path in files]
        save_report(run_dir / "gguf_export.json", {
            "schema_version": 2,
            "requested_quantizations": methods,
            "completed_quantizations": list(exports),
            "exports_complete": len(exports) == len(methods),
            "quantization": methods[0] if len(methods) == 1 else None,
            "files": [path for paths in exports.values() for path in paths],
            "exports": {key: [{"path": name, "size_bytes": Path(name).stat().st_size} for name in names] for key, names in exports.items()},
            "verification": "GGUF magic header and minimum file size; inference not tested",
        })
        for path in files:
            print(f"Saved GGUF: {path} ({path.stat().st_size / 2**30:.2f} GiB)")
    return exports


if RUN_MODE != "search":
    if EXPORT_GGUF:
        if RUN_MODE != "full":
            raise ValueError("Use a full training run before exporting a deployment candidate.")
        GGUF_DIR = RUN_DIR / "gguf"
        export_gguf_variants(
            RUN_DIR, globals().get("GGUF_QUANTIZATIONS", ["q4_k_m"]),
            lambda method: model.save_pretrained_gguf(str(GGUF_DIR), tokenizer, quantization_method=method),
            save_json,
        )
    else:
        print("GGUF export skipped. Verify saved adapter and checkpoint artifacts separately.")


## Record the result of each initial run

Before calling a run successful, review:
- `checkpoint_audit.json`: nonempty adapter files and full training state for the retained checkpoints.
- `loss_curves.png` / `loss_history.csv`: finite loss, the validation trend, and any sign of overfitting.
- `run_summary.json`: before/after validation loss, selected checkpoint, duration, GPU use, and whether this was only a smoke test.
- `validation_by_task.json` / `validation_generations.json`: JSON-task and rationale losses plus readable validation responses.

For the next experiment, use a new `RUN_NAME`; record what changed and your observed result. If validation loss is still falling, try additional epochs in a **new** run. If it rises while training loss drops, try fewer epochs or a smaller learning rate. For an out-of-memory error, start a new run with a smaller sequence length (only if all examples fit), smaller batch size, or lower LoRA rank. Do not infer hallucination accuracy or final convergence from the training loss alone.

### References

API choices were checked against the [official Unsloth Llama notebook](https://github.com/unslothai/notebooks/blob/main/nb/Llama3.2_(1B_and_3B)-Conversational.ipynb), [Unsloth Colab setup](https://unsloth.ai/docs/get-started/install/google-colab), [TRL 0.22.2 SFT documentation](https://huggingface.co/docs/trl/v0.22.2/sft_trainer), and [Transformers Trainer documentation](https://huggingface.co/docs/transformers/main_classes/trainer). GPU training and package installation must be executed and validated in your Kaggle runtime.

Save & Run All creates a new execution; verify its status and download its Output files when done. Do not assume an interrupted editor session was committed.

### Search outputs and promotion

`search_results.csv` / `search_results.json` contain every trial, including failures; `best_hyperparameters.json` contains the best successful configuration, objective, dataset fingerprint, and whether the search completed. Each `trials/trial_*/attempt_*` folder has metrics, loss curves, an adapter, and audited checkpoints. Completed trials can be skipped on search resume; interrupted trials start fresh in a new attempt folder.

To run the full experiment: attach the saved search output (which already includes its training CSV), set `RUN_MODE="full"`, a new `RUN_NAME` such as `llama32_3b_best_full_01`, `BEST_PARAMS_FILE="auto"`, and `RESUME=False`. The notebook validates model/data/task/split compatibility and applies the chosen parameters. All training tickets are used except the validation split and the excluded held-out tenant. Evaluate the resulting full model's groundedness and escalation behavior before accepting it. Screening trials are not equivalent to complete full-data runs.

Reference: [Hugging Face hyperparameter search](https://huggingface.co/docs/transformers/hpo_train). This notebook uses a small explicit grid/random loop so that fresh Unsloth adapters, bounded GPU memory, and file outputs remain visible without another search service dependency.